# `querexfuzz` project 

## Provenance
* Created new: 2025-08-27

In [1]:
%load_ext IPython.extensions.autoreload
%autoreload 2

from functools import partial
from pathlib import Path
from pprint import pprint
import logging 
import yaml 

def setup_logging(config_path):
    with config_path.open("r") as f:
        config = yaml.safe_load(f)
        logging.config.dictConfig(config)

setup_logging(Path("logconfig.yaml"))

from lark import Lark
import lark
import pandas as pd

from querexfuzz import Querexfuzz
from querexfuzz.parser import parser
import querexfuzz.parser as qfp
import querexfuzz.engine as qfe

In [7]:
parser('top 5 where year == 2025 and (name == joe or name == simon) @mod m-3:1')

10:58:35|ERROR |querexfuzz.parser[parser.py:267 parser] Failed to parse query: 'top 5 where year == 2025 and (name == joe or name == simon) @mod m-3:1.
Error: No terminal matches '(' in the current parser context, at line 1 col 30

top 5 where year == 2025 and (name == joe or name == simon) @mod m-3:
                             ^
Expected one of: 
	* TILDE
	* IDENTIFIER



ValueError: Failed to parse query: 'top 5 where year == 2025 and (name == joe or name == simon) @mod m-3:1'. Error No terminal matches '(' in the current parser context, at line 1 col 30

top 5 where year == 2025 and (name == joe or name == simon) @mod m-3:
                             ^
Expected one of: 
	* TILDE
	* IDENTIFIER


In [3]:
parser('top 10 top 20')

10:40:09|WARNING|querexfuzz.parser[parser.py:56 clause_list] re-setting top from 10 to 20


{'select': {'include': [], 'exclude': []},
 'sort': [],
 'regex': [],
 'where': None,
 'top': 20,
 'flags': [],
 'dates': [],
 'fuzzy': None}

# Pure parsing test

In [8]:
test_cases = {
 0: ('pass', '', ''),
 1: ('pass', '', '@ m'),
 2: ('pass', '', 'top 4 @ m-3'),
 3: ('fail', 'only integer time deltas', 'recent select m, y, q @ y-3:1.234'),
 4: ('pass', '', 'recent top 3 @ m'),
 5: ('pass', '', 'verbose recent top 17 @COL d-30:10 @MOD m-3'),
 6: ('pass', '', 'select roger'),
 7: ('pass', '', 'select a_col'),
 8: ('pass', '', 'select a_col2, b_col'),
 9: ('pass', '', 'select *'),
 10: ('pass', '', 'top 10 select *'),
 11: ('pass', '', 'where year == 2024'),
 12: ('pass', '', 'where year == "2024"'),
 13: ('pass', '', 'where type == "book"'),
 14: ('pass', '', 'where price <= 12.50'),
 15: ('pass', '', '! Delbaen'),
 16: ('pass', '', '! /Wang, R/ and journal ~ Annals'),
 17: ('pass', '', 'recent top 3 author ~ /Wang, R/'),
 18: ('pass', '', 'select journal author ~ /Wang, R/'),
 19: ('pass', '', 'verbose top 4 recent select journal author ~ /Wang, R/'),
 20: ('pass', '', 'top 5 select journal author ~ /Wang, R/'),
 21: ('pass', '', 'sort by journal'),
 22: ('pass', '', 'order by journal, -year'),
 23: ('pass', '', 'order by journal, -year'),
 24: ('pass', 'changing top ok but warning', 'top 6 top 26 sort by journal'),
 25: ('pass', '', 'top 7 sort by -journal'),
 26: ('pass', '', 'top 8 order by author, journal, year'),
 27: ('pass', '', 'top 9 select journal where year == 2024 order by author'),
 28: ('pass',
      '',
      'verbose top 10 select journal where year == 2024 order by -journal, '
      'author'),
 29: ('pass',
      '',
      'recent top 14  verbose select *, -c ! /Wang, R/ and journal ~ /[A-J]+/ '
      'where year == 2024 and publisher ==  "Springer" and mod > 2024'),
 30: ('pass',
      '',
      'recent top 15  verbose select *, -c ! /Wang, R/ and journal ~ [A-J]+ '
      'where year == 2024 and publisher ==  "Springer" and mod > "2024-05"'),
 31: ('pass',
      '',
      'recent top 20  verbose select *, -c ! /Wang, R/ and journal ~ [A-J]+ '
      'where year == 2024 and publisher ==  "Springer" and mod > 2024-05'),
 32: ('pass',
      '',
      'recent top 21  verbose select *, -c ! /Wang, R/ and journal ~ [A-J]+ '
      'where year == 2024 and publisher ==  "Springer" and mod > "2024-05"'),
 33: ('pass',
      '',
      'top 30 select c, d, -a, e recent ! /Wang, R/ and journal ~ [A-J]+ where '
      'year == 2024 and publisher ==  "Springer" and mod > 2024 order by name, '
      'year'),
 34: ('fail', 'obvious error', 'error'),
 35: ('pass',
      '',
      'top 40 recent where age > 30 name ~ /^A/  @ y # alice amsterdam'),
 36: ('pass', '', 'recent top 50 where author == "Wang" !Ruodu # fuzzy term'),
 37: ('pass',
      'odd order',
      'where price > 10.5 select *, -id top 20 order by price'),
 38: ('pass',
      'odd order',
      'order by -modified, created verbose recent select path where size > '
      '1000'),
 39: ('pass',
      'odd order',
      "select name, id recent where type == 'user' top 50"),
 40: ('pass',
      'odd order, unquoted regex',
      'author ~ Smith order by name select *'),
 41: ('pass', 'cols vs keywords', 'select top, order, by, where, select'),
 42: ('pass',
      'cols vs keywords',
      'where top == 1 and sort == "asc" and by == "user"'),
 43: ('pass', 'cols vs keywords', 'select y, m, d, h order by q'),
 44: ('pass', 'cols vs keywords', 'where year > y and month > m'),
 45: ('pass',
      'complex where, regex, date',
      'where price >= -99.5 and price <= 21.25 and category != "sale" and name == "a name with '
      'spaces"'),
 46: ('pass',
      'complex where, regex, date',
      'where start_date > end_date and value > 0'),
 47: ('pass',
      'complex where, regex, date',
      '! /A/ and author ~ /B/ and title ~ /C/ and journal ~ /D/'),
 48: ('pass',
      'complex where, regex, date',
      '@created q-1 @modified w-2 @accessed d-3 @expires h-4'),
 49: ('pass',
      'complex where, regex, date',
      'select id, -name, created, -updated, path order by -created, path'),
 50: ('pass',
      'formatting, minimal',
      ' \t top   10 \n   select name, id \n where \t year==2025'),
 51: ('pass', 'formatting, minimal', '# just a fuzzy search string by itself'),
 52: ('pass', 'formatting, minimal', 'select name order by name'),
 53: ('pass', 'formatting, minimal', '@ d-0'),
 54: ('fail', 'single =', 'where value = 5'),
 55: ('fail', 'keyword for identifier', 'select name, where type == "book"'),
 56: ('fail', 'trailing comma', 'top 10 order by name,'),
 57: ('fail', 'non-numeric', '@ m-5:foo')}

In [10]:
x = 10 
f'asdf {
x}'

'asdf 10'

In [21]:
parser('top 5 where year == 2025 or name == joe @mod m-3:1')

10:39:23|INFO  |querexfuzz.parser[parser.py:19 <module>] built parser.py
10:39:24|ERROR |querexfuzz.parser[parser.py:274 parser] Failed to Transform: 'top 5 where year == 2025 or name == joe @mod m-3:1'.
Error:
Error trying to process rule "where_clause_list":

tuple index out of range


ValueError: Failed to transform query: 'top 5 where year == 2025 or name == joe @mod m-3:1'. Error: Error trying to process rule "where_clause_list":

tuple index out of range

In [12]:
def run_tests(parser, test_dict, verbose=True):
    """
    Run parser tests on test_dict.

    test_dict = { (n, desc) -> test}
    
    """
    success = fail =  0
    for n, (expected, desc, query) in test_dict.items(): 
        try:
            if verbose: print(f"Query No. {n}\n{desc}")
            if verbose: print(f'{query = }\n')
            spec_dict = parser(query)
        except Exception as e:
            fail += 1
            if verbose or expected != 'fail':
                print(f'result: fail ({expected} expected) for query {n}: {query}')
        else:
            success += 1
            if verbose or expected != 'pass': 
                print(f'result: pass ({expected})\n')
            if verbose: pprint(spec_dict)
        if verbose: print('-'*40, end='\n\n')
    es = sum(i[0]=='pass' for i in test_dict.values())
    ef = len(test_dict) - es
    print(f'''
    SUMMARY:
    Pass    {success} vs expected {es}
    Fail    {fail} vs expected {ef}
    Total   {success+fail}
    Pct     {success / (success+fail):.0%}
    ''')

In [13]:
run_tests(parser, test_cases, False)


    SUMMARY:
    Pass    52 vs expected 52
    Fail    6 vs expected 6
    Total   58
    Pct     90%
    


In [14]:
run_tests(parser, test_cases, True)

Query No. 0

query = ''

result: pass (pass)

{'dates': [],
 'flags': [],
 'fuzzy': None,
 'regex': [],
 'select': {'exclude': [], 'include': []},
 'sort': [],
 'top': -1,
 'where': None}
----------------------------------------

Query No. 1

query = '@ m'

result: pass (pass)

{'dates': [{'end': 0, 'field': None, 'start': 1, 'unit': 'm'}],
 'flags': [],
 'fuzzy': None,
 'regex': [],
 'select': {'exclude': [], 'include': []},
 'sort': [],
 'top': -1,
 'where': None}
----------------------------------------

Query No. 2

query = 'top 4 @ m-3'

result: pass (pass)

{'dates': [{'end': '0', 'field': None, 'start': 3, 'unit': 'm'}],
 'flags': [],
 'fuzzy': None,
 'regex': [],
 'select': {'exclude': [], 'include': []},
 'sort': [],
 'top': 4,
 'where': None}
----------------------------------------

Query No. 3
only integer time deltas
query = 'recent select m, y, q @ y-3:1.234'

result: fail (fail expected) for query 3: recent select m, y, q @ y-3:1.234
-------------------------------------

In [15]:
# parser(test_cases[45][2])
parser('where price >= -99.5')

{'select': {'include': [], 'exclude': []},
 'sort': [],
 'regex': [],
 'where': 'price >= -99.5',
 'top': -1,
 'flags': [],
 'dates': [],
 'fuzzy': None}

In [19]:
parser(" ! /[ae]{2,}/ and version ~ [a-e]{5} ")

{'select': {'include': [], 'exclude': []},
 'sort': [],
 'regex': [('BANG', '[ae]{2,}'), ('version', '[a-e]{5}')],
 'where': None,
 'top': -1,
 'flags': [],
 'dates': [],
 'fuzzy': None}

# Simple example

In [20]:
# 1. Create a sample DataFrame
data = {
    'name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'age': [25, 30, 35, 40, 45],
    'city': ['Amsterdam', 'Berlin', 'Copenhagen', 'Berlin', 'Amsterdam'],
    'registered_date': pd.to_datetime(['2025-08-10', '2025-06-15', '2024-01-20', '2025-08-25', '2025-07-30'])
}
df = pd.DataFrame(data)
df

,name,age,city,registered_date
0,Alice,25,Amsterdam,2025-08-10
1,Bob,30,Berlin,2025-06-15
2,Charlie,35,Copenhagen,2024-01-20
3,David,40,Berlin,2025-08-25
4,Eve,45,Amsterdam,2025-07-30


In [21]:
# 2. Initialize Querexfuzz with your config file
# (Ensure config.yml is in the same directory or provide the correct path)
qflex_engine = Querexfuzz(config_path='config.yaml')

In [22]:
qflex_engine.config

QuerexfuzzConfig(base_cols=['name', 'city', 'registered_date', 'age'], date_fields=['registered_date'], default_date_field='registered_date', bang_field='name', recent_field='registered_date', fuzzy=FuzzyConfig(fields=['name', 'city'], limit=50, score_col_name='score'))

In [23]:
# 3. Attach the .querexfuzz method to your DataFrame
df = qflex_engine.attach_to(df)

In [24]:
# 4. Run queries!
print("--- People from Berlin ---")
df.querexfuzz("where city == 'Berlin'")

--- People from Berlin ---


,name,city,registered_date,age
1,Bob,Berlin,2025-06-15,30
3,David,Berlin,2025-08-25,40


In [25]:
print("\n--- Recently registered (last 1 month) ---")
df.querexfuzz("recent @m-1")


--- Recently registered (last 1 month) ---


,name,city,registered_date,age
3,David,Berlin,2025-08-25,40
0,Alice,Amsterdam,2025-08-10,25
4,Eve,Amsterdam,2025-07-30,45


In [27]:
# same without number
df.querexfuzz("recent @m")

,name,city,registered_date,age
3,David,Berlin,2025-08-25,40
0,Alice,Amsterdam,2025-08-10,25
4,Eve,Amsterdam,2025-07-30,45


In [29]:
print("\n--- Less recently registered (last 24 months but more than 4) ---")
df.querexfuzz("recent @m-24:4")


--- Less recently registered (last 24 months but more than 4) ---


,name,city,registered_date,age
2,Charlie,Copenhagen,2024-01-20,35


In [30]:
print("\n--- Fuzzy search for 'am', sorted by score ---")
df.querexfuzz("# am")


--- Fuzzy search for 'am', sorted by score ---


,name,city,registered_date,age
4,Eve,Amsterdam,2025-07-30,45
0,Alice,Amsterdam,2025-08-10,25


In [31]:
print("\n--- Complex query: top 1 from Berlin older than 35, select name and age ---")
df.querexfuzz("top 1 where city == 'Berlin' and age > 35 select name, age")


--- Complex query: top 1 from Berlin older than 35, select name and age ---


,name,city,registered_date,age
3,David,Berlin,2025-08-25,40


In [39]:
print("\n--- Bang query: names with a and e ! /[ae]{2,}/ ---")
df.querexfuzz(" ! [ae].*[ae]  ")


--- Bang query: names with a and e ! /[ae]{2,}/ ---


,name,city,registered_date,age
0,Alice,Amsterdam,2025-08-10,25
2,Charlie,Copenhagen,2024-01-20,35
4,Eve,Amsterdam,2025-07-30,45


In [33]:
df

,name,age,city,registered_date
0,Alice,25,Amsterdam,2025-08-10
1,Bob,30,Berlin,2025-06-15
2,Charlie,35,Copenhagen,2024-01-20
3,David,40,Berlin,2025-08-25
4,Eve,45,Amsterdam,2025-07-30


# Better test dataframes

In [42]:
from greater_tables.core import GT
from greater_tables.fabrications import quick_fab
fGT = partial(GT, large_ok=True)

In [61]:
testdf = quick_fab(rows=10, data_spec='ss3fid', 
                   # metric_name_spec='s3'*5,
                   metric_name_spec=['name', 'address', 'size', 'number', 'registered_date']
                  )
fGT(testdf)

i_0,name,address,size,number,registered_date
genuine,hydro,shamrock paragraph jungle,"-155,378","675,489",2015-12-27
characterize,deafness,breathed gift foreman,"315,702","268,625",2015-12-28
descriptive,malady,arcade administered commendable,"-91,992","600,497",2015-12-29
property,substitution,kilobytes modestly matches,"-50,750","160,307",2015-12-30
role,gloomy,puppet transformational ensures,"134,758","55,814",2015-12-31
goods,siemens,wretched knight albatross,"-9,031","317,549",2016-01-01
reel,therapist,surrounding physics endless,"222,804","879,182",2016-01-02
escalation,pleased,denunciation defendants replica,"45,805","85,443",2016-01-03
physically,define,crates stalks dispels,"-26,051","18,031",2016-01-04
involved,steaks,turnout priorities waster,"707,895","532,800",2016-01-05


In [62]:
q = Querexfuzz('config.yaml')
q.config

QuerexfuzzConfig(base_cols=['name', 'city', 'registered_date', 'age'], date_fields=['registered_date'], default_date_field='registered_date', bang_field='name', recent_field='registered_date', fuzzy=FuzzyConfig(fields=['name', 'city'], limit=50, score_col_name='score'))

In [66]:
testdf = q.attach_to(testdf)
testdf

metric,name,address,size,number,registered_date
i_0,,,,,
genuine,hydro,shamrock paragraph jungle,-155377.890829,675489,2015-12-27 22:22:10.823910
characterize,deafness,breathed gift foreman,315701.664315,268625,2015-12-28 22:22:10.823910
descriptive,malady,arcade administered commendable,-91991.698973,600497,2015-12-29 22:22:10.823910
property,substitution,kilobytes modestly matches,-50750.058161,160307,2015-12-30 22:22:10.823910
role,gloomy,puppet transformational ensures,134758.486135,55814,2015-12-31 22:22:10.823910
goods,siemens,wretched knight albatross,-9030.709577,317549,2016-01-01 22:22:10.823910
reel,therapist,surrounding physics endless,222803.713561,879182,2016-01-02 22:22:10.823910
escalation,pleased,denunciation defendants replica,45805.338369,85443,2016-01-03 22:22:10.823910
physically,define,crates stalks dispels,-26050.642093,18031,2016-01-04 22:22:10.823910


In [76]:
df.querexfuzz(" ! ice") #  "name ~ ^[A-B]")

,name,city,registered_date,age
0,Alice,Amsterdam,2025-08-10,25


In [78]:
df

,name,age,city,registered_date
0,Alice,25,Amsterdam,2025-08-10
1,Bob,30,Berlin,2025-06-15
2,Charlie,35,Copenhagen,2024-01-20
3,David,40,Berlin,2025-08-25
4,Eve,45,Amsterdam,2025-07-30


In [79]:
df.querexfuzz("@registered_date m-28:6")

,name,city,registered_date,age
2,Charlie,Copenhagen,2024-01-20,35


In [80]:
testdf.querexfuzz('# Berlin')

metric,name,registered_date
i_0,,


In [71]:
testdf.querexfuzz('select all where number < 800000 and number > 500000')

metric,name,registered_date
i_0,,
genuine,hydro,2015-12-27 22:22:10.823910
descriptive,malady,2015-12-29 22:22:10.823910
involved,steaks,2016-01-05 22:22:10.823910


In [72]:
testdf.querexfuzz(' ! ^[hdm] ')

metric,name,registered_date
i_0,,
genuine,hydro,2015-12-27 22:22:10.823910
characterize,deafness,2015-12-28 22:22:10.823910
descriptive,malady,2015-12-29 22:22:10.823910
physically,define,2016-01-04 22:22:10.823910


In [81]:
from rustfuzz import FuzzyMatcherMultiHi

In [83]:
FuzzyMatcherMultiHi?

Init signature: FuzzyMatcherMultiHi(strings, /)
Docstring:     
A multi-word fuzzy matcher that returns highlighting information.

Similar to `FuzzyMatcherMulti`, but the query method returns an additional
list containing the specific character indices that matched the query,
suitable for highlighting in a UI.
Type:           type
Subclasses:     

In [84]:
FuzzyMatcherMultiHi.query?

Signature: FuzzyMatcherMultiHi.query(self, query, top_k, /)
Docstring:
Finds the best fuzzy matches and returns their highlight indices.

Args:
    query (str): The search query, with words separated by spaces.
    top_k (int): The maximum number of results to return.

Returns:
    tuple[list[int], list[int], list[list[int]]]: A tuple containing three lists:
        - A list of indices of the matched strings.
        - A list of the corresponding scores.
        - A list of lists, where each inner list contains the character
          indices to highlight for that match.
Type:      method_descriptor

In [82]:
# 1. Define your list of candidate strings
candidates = [
    "A Case Study in Causal Inference",
    "Advanced Causality: Theory and Practice",
    "rust-analyzer: a language server for Rust",
    "The Rust Programming Language",
    "Practical Time-Series Analysis",
]

# 2. Create a matcher instance
matcher = FuzzyMatcherMultiHi(candidates)

In [85]:
# 3. Run a query
query_str = "rust lang"
indices, scores, highlights = matcher.query(query_str, top_k=2)
indices, scores, highlights

([2, 3],
 [174, 162],
 [[0, 1, 2, 3, 17, 18, 19, 20], [4, 5, 6, 7, 21, 22, 23, 24]])

In [90]:
# 4. Process the results
print(f"Top {len(indices)} results for query: '{query_str}'\n")

for i, score, highlight_indices in zip(indices, scores, highlights):
    candidate_string = candidates[i]

    # Use the highlight indices to format the output (e.g., for HTML)
    highlight_set = set(highlight_indices)

    highlighted_html = "".join(
        f"<mark>{char}</mark>" if i in highlight_set else char
        for i, char in enumerate(candidate_string)
    )
    highlighted_html = highlighted_html.replace("</mark><mark>", "")
    print(f"Score: {score}")
    print(f"  - Original:    '{candidate_string}'")
    print(f"  - Highlighted: {highlighted_html}\n")

Top 2 results for query: 'rust lang'

Score: 174
  - Original:    'rust-analyzer: a language server for Rust'
  - Highlighted: <mark>rust</mark>-analyzer: a <mark>lang</mark>uage server for Rust

Score: 162
  - Original:    'The Rust Programming Language'
  - Highlighted: The <mark>Rust</mark> Programming <mark>Lang</mark>uage



In [88]:
from IPython.display import HTML, display

In [91]:
display(HTML(highlighted_html))

In [ ]:
```

**Expected Output:**

```text
Top 2 results for query: 'rust lang'

Score: 308
  - Original:    'The Rust Programming Language'
  - Highlighted: The <mark>Rust</mark> Programming <mark>Lang</mark>uage

Score: 194
  - Original:    'rust-analyzer: a language server for Rust'
  - Highlighted: <mark>rust</mark>-analyzer: a <mark>lang</mark>uage server for Rust
```

-----

In [92]:
testdf = quick_fab(rows=10000, data_spec='ss20fid', 
                   # metric_name_spec='s3'*5,
                   metric_name_spec=['name', 'address', 'size', 'number', 'registered_date']
                  )
fGT(testdf.head())

i_0,name,address,size,number,registered_date
decaying,indexer,dedicate apaches cans mission cordially expedient debatable implosion prone surfacing exacerbating bushel medieval solicit saturdays mandate trey cast conducting peeking,"583,335","475,885",2024-11-06
dismay,sniff,vaunted scientists risky welsh underlying inflicting hobble eucalyptus lance unbridled lamenting jerry died abused contamination businesses well portraits airtight prosecuting,"456,941","527,237",2024-11-07
erections,categorizing,decouple gargantuan adjudicating twirling macintosh stadium outdoor ramp marginal lowers prospectus favorable unconvincing todays bedeviled mayhem specialties stuck snacks proposal,"160,336","71,729",2024-11-08
repudiate,assertions,accelerator tango writes plunger machinations reassignment harper camps smells toiling dean operative ding patched roaming synergies bristle paris purse efficiencies,"284,940","441,815",2024-11-09
shah,grinch,registering suffices leanest natured center developers astonishingly kilometer mitigation pandemonium predominantly anger mathematically divides impair technologist possession divisible carries greener,"-17,455","982,356",2024-11-10


In [93]:
q = Querexfuzz('config.yaml')
q.config

QuerexfuzzConfig(base_cols=['name', 'city', 'registered_date', 'age'], date_fields=['registered_date'], default_date_field='registered_date', bang_field='name', recent_field='registered_date', fuzzy=FuzzyConfig(fields=['name', 'city'], limit=50, score_col_name='score'))

In [95]:
testdf = q.attach_to(testdf)
testdf.head()

metric,name,address,size,number,registered_date
i_0,,,,,
decaying,indexer,dedicate apaches cans mission cordially expedi...,583334.701109,475885,2024-11-06 09:00:53.974223
dismay,sniff,vaunted scientists risky welsh underlying infl...,456941.167486,527237,2024-11-07 09:00:53.974223
erections,categorizing,decouple gargantuan adjudicating twirling maci...,160335.668631,71729,2024-11-08 09:00:53.974223
repudiate,assertions,accelerator tango writes plunger machinations ...,284940.487070,441815,2024-11-09 09:00:53.974223
shah,grinch,registering suffices leanest natured center de...,-17454.904663,982356,2024-11-10 09:00:53.974223


In [102]:
fGT(testdf.loc[testdf.querexfuzz('address ~ \\brisky\\b').index])

i_0,name,address,size,number,registered_date
dismay,sniff,vaunted scientists risky welsh underlying inflicting hobble eucalyptus lance unbridled lamenting jerry died abused contamination businesses well portraits airtight prosecuting,"456,941","527,237",2024-11-07
dissolved,fruity,prone surfacing exacerbating bushel medieval solicit saturdays mandate trey cast conducting peeking vaunted scientists risky welsh underlying inflicting hobble eucalyptus,258,"866,265",2028-05-25
tracked,ranting,trey cast conducting peeking vaunted scientists risky welsh underlying inflicting hobble eucalyptus lance unbridled lamenting jerry died abused contamination businesses,"213,490","335,607",2031-12-12
satisfaction,dockers,cordially expedient debatable implosion prone surfacing exacerbating bushel medieval solicit saturdays mandate trey cast conducting peeking vaunted scientists risky welsh,"360,312","45,624",2035-06-29
murderers,revisions,medieval solicit saturdays mandate trey cast conducting peeking vaunted scientists risky welsh underlying inflicting hobble eucalyptus lance unbridled lamenting jerry,"76,839","314,726",2039-01-15
superstition,observation,vaunted scientists risky welsh underlying inflicting hobble eucalyptus lance unbridled lamenting jerry died abused contamination businesses well portraits airtight prosecuting,"153,013","736,763",2042-08-03
wavelengths,prevails,prone surfacing exacerbating bushel medieval solicit saturdays mandate trey cast conducting peeking vaunted scientists risky welsh underlying inflicting hobble eucalyptus,"406,940","417,306",2046-02-18
advanced,deficient,trey cast conducting peeking vaunted scientists risky welsh underlying inflicting hobble eucalyptus lance unbridled lamenting jerry died abused contamination businesses,"102,739","-9,027",2049-09-06


In [ ]:
testdf.loc[testdf.querexfuzz('top 10 # risky').index]